<a href="https://colab.research.google.com/github/tayyaba315/Ai_generated_content_detector-/blob/main/FauxSight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets pillow

In [ ]:
from datasets import load_dataset
import numpy as np
import cv2

dataset = load_dataset("dragonintelligence/CIFAKE-image-dataset")
print(dataset)
print(f"Train: {len(dataset['train'])}  Test: {len(dataset['test'])}")

In [ ]:
train_labels = [s['label'] for s in dataset['train']]
test_labels  = [s['label'] for s in dataset['test']]

print("Train — Real:", train_labels.count(1), " Fake:", train_labels.count(0))
print("Test  — Real:", test_labels.count(1),  " Fake:", test_labels.count(0))

In [ ]:
IMG_SIZE = 32

data   = []
labels = []

# dataset labels: 0 = FAKE, 1 = REAL
# flipping so our convention is: 0 = Real, 1 = AI-Generated
for split in ['train', 'test']:
    for sample in dataset[split]:
        img = np.array(sample['image'].convert('RGB'))
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        data.append(img)
        labels.append(1 - sample['label'])

data   = np.array(data)
labels = np.array(labels)

print(f"Shape: {data.shape}  Real: {int((labels==0).sum())}  Fake: {int((labels==1).sum())}")

In [ ]:
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

# drop any corrupt entries
data, labels = zip(*[(img, lbl) for img, lbl in zip(data, labels) if img is not None and img.ndim == 3])
data   = np.array(data).astype('float32') / 255.0
labels = np.array(labels)

data, labels = shuffle(data, labels, random_state=42)

X_train, X_temp, y_train, y_temp = train_test_split(data, labels, test_size=0.3, random_state=42, stratify=labels)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

y_train = y_train.astype('float32')
y_val   = y_val.astype('float32')
y_test  = y_test.astype('float32')

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    Conv2D(32,  (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    Conv2D(64,  (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    Conv2D(128, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    Conv2D(256, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
BATCH_SIZE = 32
EPOCHS     = 30

rotation_layer = tf.keras.layers.RandomRotation(0.05)

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, 0.9, 1.1)
    image = rotation_layer(tf.expand_dims(image, 0))[0]
    return image, label

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train), reshuffle_each_iteration=True)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
    .repeat()
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/content/fauxsight_best.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

history = model.fit(
    train_ds,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('FauxSight – Training History', fontsize=14)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)

y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype('int32')
y_true      = y_test.astype('int32')

acc  = accuracy_score(y_true, y_pred)
mf1  = f1_score(y_true, y_pred, average='macro')

print(f"Accuracy : {acc*100:.2f}%")
print(f"Macro F1 : {mf1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=['Real', 'AI-Generated']))

fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2, figure=fig)

ax1  = fig.add_subplot(gs[0])
cm   = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
ConfusionMatrixDisplay(cm, display_labels=['Real', 'AI-Generated']).plot(ax=ax1, cmap='Blues', colorbar=False)
ax1.set_title('Confusion Matrix', fontweight='bold')
ax1.set_xlabel(f'Predicted\n\nTN={tn}  FP={fp}  FN={fn}  TP={tp}')

ax2 = fig.add_subplot(gs[1])
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc:.4f}')
ax2.plot([0,1], [0,1], color='gray', lw=1, linestyle='--', label='Random')
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.05])
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.suptitle('FauxSight – Test Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
model.save('/content/fauxsight_final.keras')
model.export('/content/fauxsight_savedmodel')
print("Saved.")

In [ ]:
def predict_image(img_path, threshold=0.5):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not read: {img_path}")

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype('float32') / 255.0
    prob = float(model.predict(np.expand_dims(img, 0), verbose=0)[0][0])

    return {
        'label':      'AI-Generated' if prob >= threshold else 'Real',
        'probability': round(prob, 4),
        'confidence':  f"{max(prob, 1 - prob) * 100:.1f}%"
    }


# test on one sample from the test set
cv2.imwrite('/tmp/test_sample.jpg', (X_test[0] * 255).astype('uint8'))
result = predict_image('/tmp/test_sample.jpg')
true_label = 'AI-Generated' if y_test[0] == 1 else 'Real'

print(f"True      : {true_label}")
print(f"Predicted : {result['label']}")
print(f"Confidence: {result['confidence']}")

In [ ]:
def predict_video(video_path, threshold=0.5, frame_skip=5):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open: {video_path}")

    probs = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_skip == 0:
            img = cv2.resize(frame, (IMG_SIZE, IMG_SIZE)).astype('float32') / 255.0
            prob = float(model.predict(np.expand_dims(img, 0), verbose=0)[0][0])
            probs.append(prob)
        frame_idx += 1

    cap.release()

    if not probs:
        return {'error': 'No frames could be extracted'}

    avg_prob = float(np.mean(probs))

    return {
        'label':           'AI-Generated' if avg_prob >= threshold else 'Real',
        'confidence':      f"{max(avg_prob, 1 - avg_prob) * 100:.1f}%",
        'avg_probability':  round(avg_prob, 4),
        'frames_analyzed':  len(probs),
        'fake_frames':      sum(1 for p in probs if p >= threshold),
        'real_frames':      sum(1 for p in probs if p < threshold),
    }


# usage example (replace with an actual video path):
# result = predict_video('/content/sample.mp4')
# print(result)